# Micro Proyecto 3 — Finetuning de Llama-3.2-1B con LoRA en RACE

**Maestría en Inteligencia Artificial — Universidad de los Andes**  
*Modelos Avanzados para el Procesamiento de Lenguaje Natural*

---

Este notebook implementa el finetuning del modelo `Llama-3.2-1B` sobre el dataset RACE
(preguntas de selección múltiple de comprensión de lectura), comparando el desempeño
del modelo base contra una versión adaptada con LoRA.

**Estrategia de evaluación.** Para cada ejemplo se calcula:

$$\text{Respuesta} = \arg\max_{s \in \{A,B,C,D\}} \log P(s \mid c)$$

donde $c$ es el contexto (prompt + artículo + pregunta + opciones). Como cada opción se
codifica como un único token (`" A"`, `" B"`, `" C"`, `" D"`), basta con un forward pass
del prompt y leer las probabilidades en la última posición.

**Decisiones técnicas clave:**
- Padding **a la derecha** en training (consistente con el cálculo de loss desplazado)
- Padding **a la izquierda** en testing (permite extraer `logits[:, -1, :]` uniformemente)
- `pad_token = eos_token` (Llama no tiene token de padding dedicado; `attention_mask` distingue)
- Mixed precision con **bf16** si está disponible (sin `GradScaler`), fallback a fp16
- LoRA aplicado a las matrices de atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`)


## 1. Setup e imports

In [ ]:
import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

import matplotlib.pyplot as plt
from tqdm.auto import tqdm


In [ ]:
# ---------------- Reproducibilidad ----------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------------- Hiperparámetros ----------------
MODEL_NAME    = "meta-llama/Llama-3.2-1B"
MAX_LENGTH    = 768          # holgura sobre lo que produce article < 800 chars
BATCH_TRAIN   = 4
BATCH_EVAL    = 16
LR            = 2e-4
EPOCHS        = 2
WARMUP_RATIO  = 0.05
GRAD_CLIP     = 1.0
WEIGHT_DECAY  = 0.01

# ---------------- GPU optimizations ----------------
# cudnn.benchmark autotunea kernels; útil siempre que los shapes sean estables
torch.backends.cudnn.benchmark = True
# TF32 solo aplica a operaciones fp32; en bf16/fp16 no tiene efecto.
# Lo dejamos activado por si alguna op cae a fp32 (ej. capas LayerNorm en algunos casos).
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# ---------------- Mixed precision ----------------
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Mixed precision: {'bf16' if USE_BF16 else 'fp16'} (AMP_DTYPE={AMP_DTYPE})")


## 2. Tokenizer

**Notas importantes:**
- Llama no tiene un token `[PAD]` dedicado. Reusamos `eos_token` como pad. La distinción
  entre padding real y EOS se hace siempre vía `attention_mask`.
- Configuramos `padding_side = "left"` como default global. Aunque nuestros `collate_fn`
  hacen el padding manualmente (e ignoran este setting), tenerlo en `"left"` es defensivo
  en caso de que algún utilitario externo (como `model.generate`) consulte el atributo.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"   # defensivo; collate_fn lo dicta de todos modos

print(f"Vocab size:    {tokenizer.vocab_size}")
print(f"BOS: {tokenizer.bos_token_id} ({tokenizer.bos_token!r})")
print(f"EOS: {tokenizer.eos_token_id} ({tokenizer.eos_token!r})")
print(f"PAD: {tokenizer.pad_token_id} ({tokenizer.pad_token!r})")


## 3. Dataset RACE

Filtramos los tres splits para conservar únicamente ejemplos cuyo artículo tenga
**menos de 800 caracteres**, según especificación del proyecto.


In [ ]:
dataset = load_dataset("ehovy/race", "all")

def short_article(example):
    return len(example["article"]) < 800

train_ds_raw = dataset["train"].filter(short_article)
val_ds_raw   = dataset["validation"].filter(short_article)
test_ds_raw  = dataset["test"].filter(short_article)

print(f"train: {len(train_ds_raw):>6}")
print(f"val:   {len(val_ds_raw):>6}")
print(f"test:  {len(test_ds_raw):>6}")

# Inspección rápida de un ejemplo
ex = train_ds_raw[0]
print("\n--- Ejemplo ---")
print(f"Article: {ex['article'][:200]}...")
print(f"Question: {ex['question']}")
print(f"Options: {ex['options']}")
print(f"Answer: {ex['answer']}")


## 4. Construcción del prompt y tokenización

El prompt termina en `"Answer:"` (sin espacio final). La respuesta se tokeniza con
**espacio adelante** (`" A"`) — esto es crucial porque en BPE el espacio es parte del
siguiente token, así que `tokenizer(" A")` y `tokenizer("A")` producen tokens distintos.

### 4.1. Tokenización para training

- Encodea prompt y answer **por separado** y concatena → evita problemas de fronteras BPE
- `add_special_tokens=True` solo en el prompt → BOS al inicio
- `add_special_tokens=False` en el answer → no duplica BOS
- EOS añadido manualmente al final → el modelo aprende a parar
- Labels: `-100` en el contexto, los ids reales en answer + EOS

### 4.2. Tokenización para testing

- Solo el prompt (terminado en `"Answer:"`) sin EOS
- No hay labels — la accuracy se calcula al vuelo durante la evaluación
- Conserva el `answer` correcto para computar accuracy


In [ ]:
def build_prompt(example):
    PROMPT_TEMPLATE = (
        "Read the article and answer the multiple-choice question "
        "by selecting the letter (A, B, C, or D) of the correct option.\n\n"
        "Article: {article}\n\n"
        "Question: {question}\n\n"
        "A) {opt_a}\n"
        "B) {opt_b}\n"
        "C) {opt_c}\n"
        "D) {opt_d}\n\n"
        "Answer:"
    )
    return PROMPT_TEMPLATE.format(
        article=example["article"].strip(),
        question=example["question"].strip(),
        opt_a=example["options"][0],
        opt_b=example["options"][1],
        opt_c=example["options"][2],
        opt_d=example["options"][3],
    )


In [ ]:
def tokenize_for_training(example, tokenizer=tokenizer):
    prompt = build_prompt(example)
    answer = " " + example["answer"].strip()  # " A" / " B" / " C" / " D"

    prompt_tokens = tokenizer(prompt, add_special_tokens=True)
    answer_tokens = tokenizer(answer, add_special_tokens=False)
    prompt_ids = prompt_tokens["input_ids"]
    answer_ids = answer_tokens["input_ids"]

    input_ids      = prompt_ids + answer_ids + [tokenizer.eos_token_id]
    attention_mask = prompt_tokens["attention_mask"] + answer_tokens["attention_mask"] + [1]
    labels         = [-100] * len(prompt_ids) + answer_ids + [tokenizer.eos_token_id]

    assert len(input_ids) == len(labels) == len(attention_mask)
    assert input_ids[-1] == tokenizer.eos_token_id
    assert input_ids[0]  == tokenizer.bos_token_id
    assert labels[len(prompt_ids)] == answer_ids[0]

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    }


In [ ]:
def tokenize_for_testing(example, tokenizer=tokenizer):
    prompt = build_prompt(example)

    prompt_tokens  = tokenizer(prompt, add_special_tokens=True)
    prompt_ids     = prompt_tokens["input_ids"]
    attention_mask = prompt_tokens["attention_mask"]

    assert len(prompt_ids) == len(attention_mask)
    assert prompt_ids[0]  == tokenizer.bos_token_id
    assert prompt_ids[-1] != tokenizer.eos_token_id  # NO debe terminar en EOS

    return {
        "input_ids":      prompt_ids,
        "attention_mask": attention_mask,
        "answer":         example["answer"].strip(),  # se preserva para computar accuracy
    }


In [ ]:
# Aplicamos tokenización a cada split.
# - train usa el formato con labels para entrenar
# - val y test usan el formato test-style (solo prompt + answer correcto)

train_ds = train_ds_raw.map(
    tokenize_for_training,
    remove_columns=train_ds_raw.column_names,
    desc="Tokenizing train",
)

val_ds_for_acc = val_ds_raw.map(
    tokenize_for_testing,
    remove_columns=val_ds_raw.column_names,
    desc="Tokenizing val (test-style)",
)

test_ds = test_ds_raw.map(
    tokenize_for_testing,
    remove_columns=test_ds_raw.column_names,
    desc="Tokenizing test",
)

# Inspección de longitudes para verificar que MAX_LENGTH es suficiente
lengths = [len(x["input_ids"]) for x in train_ds]
print(f"Train token lengths — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {np.mean(lengths):.0f}, p99: {np.percentile(lengths, 99):.0f}")
print(f"MAX_LENGTH = {MAX_LENGTH} (debería ser >= max)")


## 5. Collate functions

Padding dinámico — cada batch se pad-ea a la longitud del más largo del batch (no a
`MAX_LENGTH` global), con redondeo a múltiplo de 8 para activar Tensor Cores en bf16/fp16.

### Diferencias clave

|                | Training            | Testing             |
|----------------|---------------------|---------------------|
| Lado de padding | **derecha**        | **izquierda**       |
| Labels          | sí, con `-100` en padding | no              |
| Por qué         | El loss usa shift interno; right padding alinea respuestas al inicio | `logits[:, -1, :]` toma la próxima predicción uniformemente para todo el batch |


In [ ]:
def collate_fn_train(batch, pad_id=tokenizer.pad_token_id, pad_to_multiple_of=8):
    max_len = max(len(x["input_ids"]) for x in batch)
    if pad_to_multiple_of:
        m = pad_to_multiple_of
        max_len = ((max_len + m - 1) // m) * m

    ids, attn, lbl = [], [], []
    for x in batch:
        n   = len(x["input_ids"])
        pad = max_len - n
        # RIGHT padding
        ids .append(x["input_ids"]      + [pad_id] * pad)
        attn.append(x["attention_mask"] + [0]      * pad)
        lbl .append(x["labels"]         + [-100]   * pad)

    return {
        "input_ids":      torch.tensor(ids,  dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels":         torch.tensor(lbl,  dtype=torch.long),
    }


In [ ]:
def collate_fn_test(batch, pad_id=tokenizer.pad_token_id, pad_to_multiple_of=8):
    max_len = max(len(x["input_ids"]) for x in batch)
    if pad_to_multiple_of:
        m = pad_to_multiple_of
        max_len = ((max_len + m - 1) // m) * m

    ids, attn, answers = [], [], []
    for x in batch:
        n   = len(x["input_ids"])
        pad = max_len - n
        # LEFT padding
        ids .append([pad_id] * pad + x["input_ids"])
        attn.append([0]      * pad + x["attention_mask"])
        answers.append(x["answer"])

    return {
        "input_ids":      torch.tensor(ids,  dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "answers":        answers,   # lista de strings, no tensor
    }


## 6. DataLoaders

In [ ]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_TRAIN, shuffle=True,
    collate_fn=collate_fn_train, num_workers=2,
    pin_memory=True, drop_last=True,
)

val_loader = DataLoader(
    val_ds_for_acc, batch_size=BATCH_EVAL, shuffle=False,
    collate_fn=collate_fn_test, num_workers=1,
    pin_memory=True, drop_last=False,
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_EVAL, shuffle=False,
    collate_fn=collate_fn_test, num_workers=1,
    pin_memory=True, drop_last=False,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")


## 7. Utilidades de evaluación

Tres funciones:

1. **`get_letter_token_ids`** — recupera los token ids de `" A"`, `" B"`, `" C"`, `" D"`.
   Verifica que cada uno sea un único token.

2. **`evaluate_accuracy`** — itera el dataloader, hace forward pass del prompt,
   extrae los logits de la última posición (limpio gracias al left padding),
   y selecciona la opción con mayor log-probabilidad.

3. **`generate_examples`** — genera muestras textuales con `model.generate()` para
   inspección cualitativa. Crea internamente un mini-DataLoader con `collate_fn_test`,
   garantizando consistencia con la evaluación.


In [ ]:
def get_letter_token_ids(tokenizer):
    """Devuelve dict letra -> token_id para ' A', ' B', ' C', ' D'."""
    out = {}
    for letter in ["A", "B", "C", "D"]:
        ids = tokenizer.encode(" " + letter, add_special_tokens=False)
        assert len(ids) == 1, f"' {letter}' debería ser 1 token, dio {ids}"
        out[letter] = ids[0]
    return out

# Verificación
LETTER_IDS = get_letter_token_ids(tokenizer)
print("Token IDs de las opciones:")
for l, tid in LETTER_IDS.items():
    print(f"  ' {l}' -> {tid}")


In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, dataloader, device="cuda",
                      max_examples=None, desc="Eval"):
    """Calcula accuracy usando argmax_s log P(s|c). Asume left padding."""
    model.eval()
    letters = ["A", "B", "C", "D"]
    letter_ids = get_letter_token_ids(tokenizer)
    letter_id_t = torch.tensor([letter_ids[l] for l in letters], device=device)

    correct, total = 0, 0
    per_letter_pred = {l: 0 for l in letters}
    per_letter_true = {l: {"correct": 0, "total": 0} for l in letters}

    for batch in tqdm(dataloader, desc=desc):
        if max_examples is not None and total >= max_examples:
            break

        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        true_answers   = batch["answers"]

        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # left padding -> última posición real = -1 para todo el batch
        logits     = outputs.logits[:, -1, :]
        log_probs  = torch.log_softmax(logits.float(), dim=-1)
        scores     = log_probs[:, letter_id_t]               # [B, 4]
        preds_idx  = torch.argmax(scores, dim=1).tolist()

        for pred_idx, true_letter in zip(preds_idx, true_answers):
            pred_letter = letters[pred_idx]
            per_letter_pred[pred_letter] += 1
            per_letter_true[true_letter]["total"] += 1
            if pred_letter == true_letter:
                correct += 1
                per_letter_true[true_letter]["correct"] += 1
            total += 1

    acc = correct / total
    print(f"\n{desc} -> Accuracy: {acc:.4f}  ({correct}/{total})")
    print(f"Distribución de predicciones: {per_letter_pred}")
    print("Accuracy por letra correcta:")
    for l in letters:
        d = per_letter_true[l]
        if d["total"] > 0:
            print(f"  {l}: {d['correct']}/{d['total']} = {d['correct']/d['total']:.3f}")
    return acc


In [ ]:
@torch.no_grad()
def generate_examples(model, tokenizer, dataset, n_examples=3,
                      max_new_tokens=20, device="cuda",
                      title="", seed=42):
    """Genera respuestas para n_examples aleatorios del dataset (raw).
    Crea internamente un DataLoader con collate_fn_test para consistencia."""
    model.eval()

    n_examples = min(n_examples, len(dataset))
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(dataset), size=n_examples, replace=False).tolist()

    subset_raw = dataset.select(indices)
    subset_tok = subset_raw.map(
        tokenize_for_testing,
        remove_columns=subset_raw.column_names,
    )
    loader = DataLoader(
        subset_tok, batch_size=n_examples, shuffle=False,
        collate_fn=collate_fn_test,
    )
    batch = next(iter(loader))

    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    # Con left padding, todos los prompts terminan en la misma columna
    prompt_len      = input_ids.shape[1]
    new_tokens      = out[:, prompt_len:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    print(f"\n{'='*70}")
    print(f" Ejemplos generados — {title}")
    print(f"{'='*70}")
    for j, (idx, gen) in enumerate(zip(indices, generated_texts)):
        raw = dataset[int(idx)]
        print(f"\n--- Ejemplo {j+1} (idx={idx}) ---")
        print(f"Question : {raw['question']}")
        for k, opt in enumerate(raw["options"]):
            print(f"  {chr(65+k)}) {opt}")
        print(f"Correct  : {raw['answer']}")
        print(f"Generated: {gen!r}")


## 8. Carga y evaluación del modelo base

Cargamos `Llama-3.2-1B` directamente en `bf16`/`fp16` para reducir memoria. La evaluación
del modelo base es nuestra **línea base** sobre la que mediremos la mejora del finetuning.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=AMP_DTYPE,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros del modelo: {n_params/1e6:.1f}M")
print(f"Memoria aprox: {n_params * (2 if AMP_DTYPE != torch.float32 else 4) / 1e9:.2f} GB")


In [ ]:
# Evaluación del modelo base sobre test
acc_base = evaluate_accuracy(model, test_loader, device=device, desc="Test base")


In [ ]:
# Generación de ejemplos cualitativos del modelo base
generate_examples(model, tokenizer, test_ds_raw, n_examples=3,
                  title="Modelo base", max_new_tokens=15)


## 9. Configuración de LoRA

Aplicamos LoRA a las cuatro proyecciones de atención. Con `r=8` y `alpha=16`, el número
de parámetros entrenables es ~0.1% del total — finetuning extremadamente eficiente.

| Hyperparam     | Valor   | Justificación |
|----------------|---------|---------------|
| `r`            | 8       | Rango bajo, suficiente para tarea cerrada (4 letras) |
| `lora_alpha`   | 16      | Escala = alpha/r = 2 |
| `lora_dropout` | 0.05    | Regularización ligera |
| `target_modules` | q,k,v,o proj | Atención completa |


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 10. Entrenamiento

### Configuración

- **Optimizer**: AdamW solo sobre parámetros entrenables (los de LoRA)
- **Scheduler**: linear con warmup del 5% del total de steps
- **Mixed precision**: bf16 (preferido) o fp16+`GradScaler`
- **Validación**: accuracy al final de cada epoch sobre `val_loader`
- **Checkpointing**: guarda el mejor modelo según `val_acc`

### Sobre `GradScaler`

`GradScaler` solo es necesario con **fp16**, donde gradientes muy pequeños hacen
underflow a cero. **bf16** tiene el mismo rango dinámico que fp32, así que no requiere
scaler. El código maneja ambos casos.


In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# fp16 requiere GradScaler; bf16 no
scaler = None if USE_BF16 else GradScaler()

print(f"Total steps:  {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"GradScaler:   {'No (bf16)' if scaler is None else 'Sí (fp16)'}")


In [ ]:
history = {
    "step": [], "train_loss": [],
    "val_step": [], "val_acc": [],
}
best_val_acc = -1.0
best_ckpt_path = "best_model_lora.pt"
log_every = 50

global_step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in pbar:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss

        if scaler is not None:
            # fp16 path: escala loss antes del backward
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                GRAD_CLIP,
            )
            scaler.step(optimizer)
            scaler.update()
        else:
            # bf16 path: directo
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                GRAD_CLIP,
            )
            optimizer.step()

        scheduler.step()
        global_step += 1
        epoch_losses.append(loss.item())

        if global_step % log_every == 0:
            recent = epoch_losses[-log_every:]
            avg_loss = sum(recent) / len(recent)
            history["step"].append(global_step)
            history["train_loss"].append(avg_loss)
            pbar.set_postfix({
                "loss": f"{avg_loss:.4f}",
                "lr":   f"{scheduler.get_last_lr()[0]:.2e}",
            })

    # Validación al final de cada epoch
    val_acc = evaluate_accuracy(
        model, val_loader, device=device,
        desc=f"Val epoch {epoch+1}",
    )
    history["val_step"].append(global_step)
    history["val_acc"].append(val_acc)

    # Guardar el mejor checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch":            epoch + 1,
            "val_acc":          val_acc,
            "global_step":      global_step,
            "lora_config":      lora_config.to_dict(),
        }, best_ckpt_path)
        print(f"✓ Nuevo mejor val_acc: {val_acc:.4f} → checkpoint guardado")

print(f"\nMejor val_acc durante entrenamiento: {best_val_acc:.4f}")


## 11. Evaluación final del modelo finetuneado

Cargamos el mejor checkpoint (según `val_acc`) y evaluamos sobre el conjunto de **test**.


In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Cargado checkpoint: epoch {ckpt['epoch']}, val_acc {ckpt['val_acc']:.4f}")

acc_ft = evaluate_accuracy(model, test_loader, device=device, desc="Test finetuneado")


In [ ]:
generate_examples(model, tokenizer, test_ds_raw, n_examples=3,
                  title="Modelo finetuneado", max_new_tokens=15)


## 12. Análisis y comparación de resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Train loss
axes[0].plot(history["step"], history["train_loss"], color="steelblue")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Train loss")
axes[0].set_title("Pérdida de entrenamiento")
axes[0].grid(True, alpha=0.3)

# Val accuracy por epoch
axes[1].plot(history["val_step"], history["val_acc"], "o-", color="seagreen", markersize=8)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Val accuracy")
axes[1].set_title("Validation accuracy")
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

# Comparación final
bars = axes[2].bar(["Base", "Finetuneado"], [acc_base, acc_ft],
                    color=["lightcoral", "seagreen"], edgecolor="black")
axes[2].set_ylabel("Test accuracy")
axes[2].set_title(f"Mejora: +{(acc_ft - acc_base)*100:.2f} puntos")
axes[2].set_ylim(0, max(acc_base, acc_ft) * 1.25)
for bar, v in zip(bars, [acc_base, acc_ft]):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                 f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("comparacion.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"\n=== Resultados finales ===")
print(f"  Test accuracy base:        {acc_base:.4f}")
print(f"  Test accuracy finetuneado: {acc_ft:.4f}")
print(f"  Mejora absoluta:           +{(acc_ft - acc_base)*100:.2f} pp")
print(f"  Mejora relativa:           +{(acc_ft - acc_base)/acc_base*100:.1f}%")


## 13. Conclusiones

**A completar tras ejecutar el notebook**, considerando:

- Diferencia entre accuracy base y finetuneada
- Evolución de val_acc a lo largo del entrenamiento (¿hay signos de overfitting?)
- Análisis de la distribución de predicciones por letra (¿hay sesgo hacia alguna opción?)
- Calidad cualitativa de las generaciones del modelo finetuneado vs el base

### Observaciones técnicas relevantes

- **LoRA**: con solo ~0.1% de parámetros entrenables se logra una mejora significativa,
  validando la eficiencia del método.
- **Mixed precision**: bf16 simplifica el código (sin GradScaler) y es estable en
  hardware Ampere+.
- **Padding asimétrico**: right en training, left en testing — esta distinción es
  crítica para que `logits[:, -1, :]` funcione correctamente en evaluación batched.
